In [ ]:
!pip uninstall -y mcp fastmcp langchain-mcp-adapters langgraph langchain-google-genai
!pip install -q "mcp>=1.0.0,<2.0.0" mcp-types langgraph langchain-google-genai nest_asyncio

Found existing installation: langgraph 1.2.11
Uninstalling langgraph-1.2.11:
  Successfully uninstalled langgraph-1.2.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency c

In [ ]:
!pip install -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os
import csv
import asyncio
import nest_asyncio
from google.colab import userdata

from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

nest_asyncio.apply()

# 1. API & Model Setup
api_key = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("O_R_A_K")

if not api_key:
    try:
        api_key = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        pass

os.environ["OPENROUTER_API_KEY"] = api_key

llm = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

# 2. Expense Tools
CSV_FILE = "expenses.csv"

def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as f:
            csv.writer(f).writerow(["Item", "Amount", "Category"])

def add_expense(item: str, amount: float, category: str) -> str:
    _initialize_csv()
    with open(CSV_FILE, mode='a', newline='') as f:
        csv.writer(f).writerow([item, amount, category])
    return f"Logged: {item} - ${amount} ({category})"

def get_expenses() -> str:
    _initialize_csv()
    with open(CSV_FILE, mode='r') as f:
        rows = list(csv.reader(f))
    return "No expenses." if len(rows) <= 1 else "\n".join([", ".join(r) for r in rows[1:]])

# 3. Wrap Tools & Build Agent
mcp_tools = [
    StructuredTool.from_function(
        func=add_expense,
        name="add_expense",
        description="Logs an expense."
    ),
    StructuredTool.from_function(
        func=get_expenses,
        name="get_expenses",
        description="Gets logged expenses."
    )
]

agent = create_agent(llm, mcp_tools)

# 4. Helper to extract raw text string
def get_clean_text(content) -> str:
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "".join([
            b.get("text", "")
            for b in content
            if isinstance(b, dict) and b.get("type") == "text"
        ])

    return str(content)

# 5. Execution
async def run_agentic_workflow():

    # Expense 1
    res1 = await agent.ainvoke({
        "messages": [
            ("user", "I bought a pizza for $12.50. Category is Food.")
        ]
    })
    print(get_clean_text(res1["messages"][-1].content))

    # Expense 2
    res2 = await agent.ainvoke({
        "messages": [
            ("user", "I bought coffee for $4.50. Category is Food.")
        ]
    })
    print(get_clean_text(res2["messages"][-1].content))

    # Expense 3
    res3 = await agent.ainvoke({
        "messages": [
            ("user", "I paid $20 for petrol. Category is Transport.")
        ]
    })
    print(get_clean_text(res3["messages"][-1].content))

    # Expense 4
    res4 = await agent.ainvoke({
        "messages": [
            ("user", "I bought a programming book for $35. Category is Education.")
        ]
    })
    print(get_clean_text(res4["messages"][-1].content))

    # Expense 5
    res5 = await agent.ainvoke({
        "messages": [
            ("user", "I spent $15 on a movie ticket. Category is Entertainment.")
        ]
    })
    print(get_clean_text(res5["messages"][-1].content))

    # Show all expenses
    res6 = await agent.ainvoke({
        "messages": [
            ("user", "Show me all expenses logged so far.")
        ]
    })
    print(get_clean_text(res6["messages"][-1].content))


asyncio.run(run_agentic_workflow())

Your expense has been logged: **pizza – $12.50 (Food)**.
Your coffee purchase has been logged: **$4.50 – Food**.
Your expense has been logged: **petrol – $20.00** under the **Transport** category. Let me know if you’d like to add more expenses or view your totals!
Your expense has been logged successfully! 

**Details:**
- Item: Programming book
- Amount: $35.00
- Category: Education
Your expense has been recorded: **movie ticket – $15.00 (Entertainment)**. Let me know if you’d like to add more expenses or view your total spending!
Here are all the expenses that have been logged so far:

| Item                | Amount (€) | Category   |
|---------------------|------------|------------|
| pizza               | 12.50      | Food       |
| pizza               | 12.50      | Food       |
| coffee              | 4.50       | Food       |
| petrol              | 20.00      | Transport  |
| programming book    | 35.00      | Education  |
| movie ticket        | 15.00      | Entertainment |
| 